In [1]:
from pathlib import Path
import os
import sys

import mne
from pytep import apply_sound, apply_sspsir

project_root = next(
    path for path in [Path.cwd(), *Path.cwd().parents]
    if (path / "pyproject.toml").exists()
)

os.chdir(project_root)

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

%matplotlib qt

In [ ]:
from modules.decode_trigger import decode_8bit_trigger, convert_dict_trigger
from modules.events import get_events_tms_per_task
from modules.preprocessing import fix_stim_artifact_cubic

In [10]:
raw_data = mne.io.read_raw_bdf(r"data/raw/V1.bdf", preload=True)

Extracting BDF parameters from data/raw/V1.bdf...
Setting channel info structure...
Creating raw.info structure...
Reading 0 ... 12594999  =      0.000 ...  2519.000 secs...


In [ ]:
emg_ch_names = ["EMG L", "EMG R"]
eog_ch_names = ["EOG"]
raw_data.set_channel_types({ch: "emg" for ch in emg_ch_names})
raw_data.set_channel_types({'EOG':'eog'})

montage = mne.channels.make_standard_montage("standard_1020",head_size='auto')
raw_data.set_montage(montage)

In [12]:
events, event_id = mne.events_from_annotations(raw_data)
event_id = raw_data.event_id = convert_dict_trigger(event_id, decode_8bit_trigger)

Used Annotations descriptions: [np.str_('8Bit 1'), np.str_('8Bit 10'), np.str_('8Bit 2'), np.str_('8Bit 3'), np.str_('8Bit 4'), np.str_('8Bit 5'), np.str_('Stimulus A')]


In [ ]:
raw_data.plot(scalings={"eeg": 100e-6, "emg": 500e-6, "eog": 200e-6}, n_channels=10)

In [ ]:
bad_ch =['F7', 'F5', 'P7', 'CPz', 'Oz', 'Iz', 'PO4', 'PO8', 'O2', "P6", "P8", "TP8", "C6", "TP10", "TP9", "F6", "FT8", "T8", "T7", "F3", "FC5", "FC3", "C3", "C1"]
raw_data.drop_channels(bad_ch)

In [ ]:
events_tms, events_id_tms = get_events_tms_per_task(events, event_id)

In [ ]:
raw_data = fix_stim_artifact_cubic(
    inst=raw_data, 
    events=events,
    event_id=7,
    tmin=-0.004,
    tmax=0.018,
    pre_window=0.010
    post_window=0.010
)

In [ ]:
raw_data.notch_filter(freqs=[60, 120], fir_design='firwin')

In [ ]:
epochs_left = mne.Epochs(
    raw_data,
    events=events_tms,
    event_id=events_id_tms,
    tmin=-0.8,
    tmax=0.8,
    baseline=(-0.25,-0.1),
    preload=True,
    detrend=1,
)

In [ ]:
epochs_left.average().plot()

In [ ]:
ica = mne.preprocessing.ICA(n_components=20, random_state=97, max_iter=800)
ica.fit(epochs_left)

In [ ]:
ica.plot_sources(epochs_left, show_scrollbars=True)
ica.plot_components(inst=epochs_left)

In [ ]:
ica.exclude = [0,1]
epochs_clean_left = ica.apply(epochs_left.copy())

Applying ICA to Epochs instance
    Transforming to ICA space (20 components)
    Zeroing out 2 ICA components
    Projecting back using 40 PCA components


C:\Users\marci\AppData\Local\Temp\ipykernel_43936\2884354493.py:2: RuntimeWarning: The data you passed to ICA.apply() was baseline-corrected. Please note that ICA can introduce DC shifts, therefore you may wish to consider baseline-correcting the cleaned data again.
  epochs_clean_left = ica.apply(epochs_clean_left.copy())


In [ ]:
epochs_clean_left.apply_baseline(baseline=(-0.250, -0.1))

In [ ]:
epochs_clean_left = apply_sound(epochs_clean_left)

In [ ]:
epochs_clean_left.set_eeg_reference("average", projection=True)

In [ ]:
epochs_clean_left = apply_sspsir(epochs_clean_left)

In [ ]:
#Linear interpolation of the artifact again

In [ ]:
epochs_clean_left.filter(l_freq=1, h_freq=80, fir_design='firwin')

In [ ]:
epochs_clean_left.average().plot()

In [27]:
evoked = epochs_clean_left.average()  # média de todas as épocas

In [ ]:
evoked.apply_baseline(baseline=(-0.250, -0.05))

In [ ]:
evoked.plot(picks='FC1')